### Ablation Study 3B

In [1]:
import os
import json
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             matthews_corrcoef, roc_auc_score, confusion_matrix, roc_curve)

FEATURES_PATH = "DB_features/features.csv"
MODELS_DIR = "models_final_E2"
RESULTS_DIR = "results_final_E2"
SEED = 42

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [2]:
df = pd.read_csv(FEATURES_PATH)

# feature groups
url_cols = [c for c in df.columns if c.startswith("url_")]
html_cols = [c for c in df.columns if c.startswith("html_")]
js_cols = [c for c in df.columns if c.startswith("js_") or c.startswith("tfidf_bpe_")]
feature_cols = url_cols + html_cols + js_cols
print("url:", len(url_cols), "| html:", len(html_cols), "| js:", len(js_cols), "| total:", len(feature_cols))

df[feature_cols] = df[feature_cols].fillna(0)

# use the train/test split that is already in the csv
train_df = df[df["split"] == "train"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)
y_train = train_df["label"].values
y_test = test_df["label"].values
print("train:", len(train_df), "| test:", len(test_df))

url: 12 | html: 14 | js: 254 | total: 280
train: 21651 | test: 5413


In [3]:
with open(f"{RESULTS_DIR}/best_params_E2.json") as f:
    best = json.load(f)

feature_sets = {
    "url_only": url_cols,
    "html_only": html_cols,
    "js_only": js_cols,
    "no_url": html_cols + js_cols,
    "no_html": url_cols + js_cols,
    "no_js": url_cols + html_cols,
}
{name: len(cols) for name, cols in feature_sets.items()}

{'url_only': 12,
 'html_only': 14,
 'js_only': 254,
 'no_url': 268,
 'no_html': 266,
 'no_js': 26}

In [4]:
results = []

for set_name, cols in feature_sets.items():
    # scaler fitted on the training split of this feature set only
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(train_df[cols])
    X_test = scaler.transform(test_df[cols])

    models = {
        "decision_tree": DecisionTreeClassifier(random_state=SEED, **best["decision_tree"]),
        "random_forest": RandomForestClassifier(random_state=SEED, n_jobs=-1, **best["random_forest"]),
        "xgboost": XGBClassifier(random_state=SEED, n_jobs=-1, eval_metric="logloss", **best["xgboost"]),
        "stacking": StackingClassifier(
            estimators=[("rf", RandomForestClassifier(random_state=SEED, n_jobs=-1, **best["random_forest"])),
                        ("xgb", XGBClassifier(random_state=SEED, n_jobs=-1, eval_metric="logloss", **best["xgboost"]))],
            final_estimator=LogisticRegression(solver="saga", max_iter=5000, random_state=SEED,
                                               C=best["stacking"]["final_estimator__C"],
                                               l1_ratio=best["stacking"]["final_estimator__l1_ratio"]),
            cv=5),
    }

    for name, model in models.items():
        # training time
        t0 = time.perf_counter()
        model.fit(X_train, y_train)
        train_time = time.perf_counter() - t0

        # inference time on the whole test set
        model.predict(X_test[:64])
        runs = []
        for _ in range(5):
            t0 = time.perf_counter()
            y_pred = model.predict(X_test)
            runs.append(time.perf_counter() - t0)
        infer_batch = float(np.median(runs))

        # inference time for a single sample
        single = []
        for i in range(min(200, len(X_test))):
            t0 = time.perf_counter()
            model.predict(X_test[i:i + 1])
            single.append(time.perf_counter() - t0)

        y_score = model.predict_proba(X_test)[:, 1]
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
        results.append({
            "feature_set": set_name,
            "model": name,
            "n_features": len(cols),
                "n_train": len(y_train),
                "n_test": len(y_test),
                "train_time_s": train_time,
                "infer_test_set_s": infer_batch,
                "infer_ms_per_sample": 1000 * infer_batch / len(X_test),
                "infer_single_row_ms": 1000 * float(np.median(single)),
                "accuracy": accuracy_score(y_test, y_pred),
                "precision": precision_score(y_test, y_pred),
                "recall": recall_score(y_test, y_pred),
                "macro_f1": f1_score(y_test, y_pred, average="macro"),
                "mcc": matthews_corrcoef(y_test, y_pred),
                "roc_auc": roc_auc_score(y_test, y_score),
                "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        })
        print(set_name, "|", name, "| train", round(train_time, 2), "s | Macro F1", round(f1_score(y_test, y_pred), 4))

ablation_df = pd.DataFrame(results)
ablation_df.to_csv(f"{RESULTS_DIR}/ablation_results_E2.csv", index=False)
ablation_df

url_only | decision_tree | train 0.01 s | Macro F1 0.5943
url_only | random_forest | train 1.0 s | Macro F1 0.5918
url_only | xgboost | train 0.13 s | Macro F1 0.5975
url_only | stacking | train 7.68 s | Macro F1 0.6041
html_only | decision_tree | train 0.09 s | Macro F1 0.5911
html_only | random_forest | train 4.71 s | Macro F1 0.6338
html_only | xgboost | train 0.15 s | Macro F1 0.6418
html_only | stacking | train 25.68 s | Macro F1 0.6461
js_only | decision_tree | train 0.37 s | Macro F1 0.5863
js_only | random_forest | train 84.12 s | Macro F1 0.6492
js_only | xgboost | train 0.94 s | Macro F1 0.6614
js_only | stacking | train 244.97 s | Macro F1 0.6609
no_url | decision_tree | train 0.43 s | Macro F1 0.6436
no_url | random_forest | train 56.61 s | Macro F1 0.6659
no_url | xgboost | train 0.94 s | Macro F1 0.6823
no_url | stacking | train 289.26 s | Macro F1 0.6839
no_html | decision_tree | train 0.41 s | Macro F1 0.6739
no_html | random_forest | train 61.02 s | Macro F1 0.7013
no_

,feature_set,model,n_features,n_train,n_test,train_time_s,infer_test_set_s,infer_ms_per_sample,infer_single_row_ms,accuracy,precision,recall,macro_f1,mcc,roc_auc,tn,fp,fn,tp
0,url_only,decision_tree,12,21651,5413,0.011474,0.000116,0.000022,0.023750,0.754110,0.771971,0.483152,0.708961,0.455395,0.785472,3107,288,1043,975
1,url_only,random_forest,12,21651,5413,1.001250,0.050424,0.009315,13.435896,0.742841,0.724856,0.500000,0.702043,0.428366,0.762433,3012,383,1009,1009
2,url_only,xgboost,12,21651,5413,0.133931,0.000852,0.000157,0.147208,0.758544,0.789260,0.480674,0.712512,0.466771,0.795229,3136,259,1048,970
3,url_only,stacking,12,21651,5413,7.684999,0.039957,0.007382,13.691667,0.760022,0.784640,0.491080,0.715959,0.469847,0.795201,3123,272,1027,991
4,html_only,decision_tree,14,21651,5413,0.092002,0.000290,0.000053,0.045229,0.702753,0.606677,0.576313,0.678806,0.358145,0.762679,2641,754,855,1163
5,html_only,random_forest,14,21651,5413,4.709200,0.050385,0.009308,13.344542,0.753002,0.708512,0.573340,0.723729,0.456314,0.808646,2919,476,861,1157
6,html_only,xgboost,14,21651,5413,0.148452,0.000883,0.000163,0.150187,0.753002,0.698542,0.593657,0.726674,0.458746,0.820327,2878,517,820,1198
7,html_only,stacking,14,21651,5413,25.684657,0.040889,0.007554,13.688605,0.760022,0.717483,0.587711,0.732297,0.472621,0.823336,2928,467,832,1186
8,js_only,decision_tree,254,21651,5413,0.371512,0.000436,0.000081,0.026395,0.709034,0.623812,0.553023,0.680949,0.364714,0.748543,2722,673,902,1116
9,js_only,random_forest,254,21651,5413,84.122085,0.050864,0.009397,13.313854,0.738038,0.648221,0.650149,0.720081,0.440164,0.795595,2683,712,706,1312


In [5]:
order = ["url_only", "html_only", "js_only", "no_url", "no_html", "no_js"]
model_order = ["decision_tree", "random_forest", "xgboost", "stacking"]
for metric in ["macro_f1", "mcc", "accuracy"]:
    print(metric)
    display(ablation_df.pivot(index="feature_set", columns="model", values=metric).loc[order, model_order].round(4))

macro_f1


model,decision_tree,random_forest,xgboost,stacking
feature_set,,,,
url_only,0.7090,0.7020,0.7125,0.7160
html_only,0.6788,0.7237,0.7267,0.7323
js_only,0.6809,0.7201,0.7283,0.7293
no_url,0.7082,0.7495,0.7569,0.7594
no_html,0.7522,0.7713,0.7903,0.7943
no_js,0.7375,0.7895,0.7913,0.7969


mcc


model,decision_tree,random_forest,xgboost,stacking
feature_set,,,,
url_only,0.4554,0.4284,0.4668,0.4698
html_only,0.3581,0.4563,0.4587,0.4726
js_only,0.3647,0.4402,0.4566,0.4585
no_url,0.4181,0.5106,0.5187,0.5253
no_html,0.5114,0.5475,0.5835,0.5911
no_js,0.4767,0.5814,0.5853,0.5961


accuracy


model,decision_tree,random_forest,xgboost,stacking
feature_set,,,,
url_only,0.7541,0.7428,0.7585,0.7600
html_only,0.7028,0.7530,0.7530,0.7600
js_only,0.7090,0.7380,0.7447,0.7465
no_url,0.7225,0.7774,0.7798,0.7831
no_html,0.7770,0.7927,0.8084,0.8117
no_js,0.7589,0.8073,0.8092,0.8140
